# Day 0 — Self-learning technical onboarding, dataset orientation, and domain-validity terminology

Welcome to the **AI for Nuclear Thermal Hydraulics Summer School**.

This Day 0 notebook is a **self-learning preparation pack**. Its purpose is to help you confirm that:

- your Python environment and required packages are working;
- the notebook and dataset folders are visible from the expected locations;
- the CSV data files can be loaded successfully;
- you understand the basic structure of the case-study data;
- you are familiar with the key terms used throughout the school:
  - **training domain**
  - **prediction domain**
  - **unsafe / out-of-domain region**

By the end of this notebook, you should have:

1. Run a basic environment check;
2. Identified the available CSV files;
3. Loaded at least one dataset successfully;
4. Produced one simple data plot;
5. Confirmed that your setup is ready for Day 1.

## 1. Imports and basic setup

Run this cell first. It checks that the core packages needed for the school can be imported.

In [ ]:
from pathlib import Path
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Python executable :', sys.executable)
print('Python version    :', platform.python_version())
print('Platform          :', platform.platform())
print('\nCore imports succeeded.')

## 2. Locate the notebook and dataset folders

This notebook is expected to live in:

`./ai4nth-summer-school/course_materials/day0_onboarding`

and the CSV files are expected in:

`./ai4nth-summer-school/datasets`

So, relative to this notebook, the dataset folder should be found at:

`../../datasets`

The cell below resolves those paths and checks that they exist.


In [ ]:
notebook_dir = Path.cwd().resolve()
bundle_root = notebook_dir.parents[1]
data_dir = bundle_root / 'datasets'
environment_dir = bundle_root / 'environment'

print('Notebook directory :', notebook_dir)
print('Bundle root        :', bundle_root)
print('Dataset directory  :', data_dir)
print('Environment dir    :', environment_dir)

assert data_dir.exists(), f'Dataset directory not found: {data_dir}'
print('\nPASS: dataset directory exists.')


## 3. List the available CSV files

This cell finds all `.csv` files in the dataset folder. If your dataset includes separate files for different domains or splits, they should appear here.


In [ ]:
csv_files = sorted(data_dir.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {data_dir}')

print(f'Found {len(csv_files)} CSV file(s):\n')
for i, f in enumerate(csv_files, start=1):
    print(f'{i:2d}. {f.name}')

## 4. Quick file categorisation (optional helper)

If the file names contain words such as `train`, `predict`, `prediction`, `unsafe`, `ood`, or `test`, this helper cell tries to classify them. This is only a convenience check; the real interpretation depends on the dataset documentation.

In [ ]:
def guess_domain_label(filename: str) -> str:
    name = filename.lower()
    if 'train' in name:
        return 'training domain'
    if 'predict' in name or 'prediction' in name:
        return 'prediction domain'
    if 'unsafe' in name or 'ood' in name or 'out_of_domain' in name or 'out-of-domain' in name:
        return 'unsafe / out-of-domain region'
    if 'test' in name or 'valid' in name or 'val' in name:
        return 'test / validation-like split'
    return 'unclassified from filename'

summary_rows = []
for f in csv_files:
    summary_rows.append({'file': f.name, 'guessed_role': guess_domain_label(f.name)})

file_summary = pd.DataFrame(summary_rows)
display(file_summary)

## 5. Load one dataset

Choose the file you want to inspect by editing the `selected_file` variable below.

By default, the notebook loads the first CSV file in the folder.

In [ ]:
selected_file = csv_files[0].name  # Change this if you want to inspect a different file
selected_path = data_dir / selected_file

df = pd.read_csv(selected_path)

print('Loaded file :', selected_file)
print('Shape       :', df.shape)
display(df.head())

## 6. Inspect the dataset structure

This helps you check column names, data types, and missing values.

In [ ]:
print('Column names:')
for c in df.columns:
    print('-', c)

print('\nData types:')
display(df.dtypes.to_frame('dtype'))

missing = df.isna().sum()
display(missing.to_frame('missing_values'))

## 7. Basic descriptive summary

This cell summarises the numeric columns in the selected dataset.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

if numeric_df.empty:
    print('No numeric columns were found in this file.')
else:
    display(numeric_df.describe().T)

## 8. Make one simple data plot

This is the Day 0 plotting check. It confirms that plotting works and helps you inspect the selected dataset.

The cell below automatically chooses the first two numeric columns for a simple scatter plot. If only one numeric column exists, it will make a histogram instead.

In [ ]:
num_cols = list(numeric_df.columns)

plt.figure(figsize=(6, 4))

if len(num_cols) >= 2:
    x_col, y_col = num_cols[6], num_cols[9]
    plt.scatter(df[x_col], df[y_col], alpha=0.7)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f'Scatter plot: {x_col} vs {y_col}')
elif len(num_cols) == 1:
    x_col = num_cols[0]
    plt.hist(df[x_col].dropna(), bins=20)
    plt.xlabel(x_col)
    plt.ylabel('Count')
    plt.title(f'Histogram: {x_col}')
else:
    plt.text(0.5, 0.5, 'No numeric columns available for plotting', ha='center', va='center')
    plt.axis('off')

plt.tight_layout()
plt.show()
print('PASS: plotting works.')

## 9. Domain-validity terminology used in the school

The following terms will be used throughout the summer school:

- **Training domain**: the region of input space represented by the data used to fit the model.
- **Prediction domain**: the region where we plan to apply the model and where we hope the model remains reliable.
- **Unsafe / out-of-domain region**: inputs that lie too far from the training data or outside the region where the model has been validated.

In engineering ML, especially for thermal-hydraulics problems, a model can appear accurate on average while still being unreliable in poorly covered or extrapolative regions. For this reason, later sessions will repeatedly emphasise:

- data coverage;
- interpolation versus extrapolation;
- uncertainty awareness;
- and conditions under which a model should be used with caution or not used at all.

## 10. Optional check: load all CSV files and compare shapes

This optional helper gives a quick overview of all CSV files in the dataset folder.


In [ ]:
rows = []
for f in csv_files:
    try:
        temp_df = pd.read_csv(f)
        rows.append({
            'file': f.name,
            'rows': temp_df.shape[0],
            'columns': temp_df.shape[1],
            'guessed_role': guess_domain_label(f.name),
        })
    except Exception as e:
        rows.append({
            'file': f.name,
            'rows': 'ERROR',
            'columns': 'ERROR',
            'guessed_role': str(e),
        })

overview_df = pd.DataFrame(rows)
display(overview_df)

## 11. Final readiness checklist

You are ready for Day 1 if you can answer **yes** to all of the following:

- I can run notebook cells without errors.
- I can locate the `datasets` folder.
- I can list the available CSV files.
- I can load at least one CSV file successfully.
- I can generate at least one simple plot.
- I understand the meaning of training domain, prediction domain, and unsafe / out-of-domain region.

If any of these steps fail, please contact the summer school organisers or TAs before Day 1.

In [ ]:
print('Day 0 notebook completed successfully.')
print('If the cells above have run without errors, your setup is ready for Day 1.')